In [1]:
import subprocess
subprocess.run(['pip', 'install', 'flask', 'opencv-python-headless', 'numpy', 'requests', '-q'], capture_output=True)
print('All libraries installed!')

All libraries installed!


In [ ]:
import os
import threading
import base64
import numpy as np
import cv2
import requests
from flask import Flask, request, jsonify, render_template_string

app = Flask(__name__)

HAAR_URL = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_fullbody.xml"
HAAR_PATH = "haarcascade_fullbody.xml"

if not os.path.exists(HAAR_PATH):
    r = requests.get(HAAR_URL)
    with open(HAAR_PATH, 'wb') as f:
        f.write(r.content)

body_cascade = cv2.CascadeClassifier(HAAR_PATH)

HTML_PAGE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Animal Herd Detection</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: Arial, sans-serif; background: #0d1117; color: #e6edf3; }
        .header { background: linear-gradient(135deg, #1f6feb, #388bfd); padding: 20px 30px; }
        .header h1 { font-size: 26px; }
        .header p  { font-size: 13px; opacity: 0.85; margin-top: 4px; }
        .container { display: flex; gap: 20px; padding: 24px; flex-wrap: wrap; }
        .left  { flex: 1; min-width: 320px; }
        .right { flex: 1; min-width: 320px; }
        .card { background: #161b22; border: 1px solid #30363d; border-radius: 10px; padding: 20px; margin-bottom: 20px; }
        .card h2 { font-size: 16px; margin-bottom: 14px; color: #58a6ff; }
        .upload-area { border: 2px dashed #30363d; border-radius: 8px; padding: 30px; text-align: center; cursor: pointer; transition: 0.2s; }
        .upload-area:hover { border-color: #58a6ff; background: #0d1117; }
        .upload-area input { display: none; }
        .upload-area label { cursor: pointer; color: #58a6ff; font-size: 14px; }
        .btn { width: 100%; padding: 12px; background: #1f6feb; color: white; border: none; border-radius: 8px; font-size: 15px; cursor: pointer; margin-top: 12px; }
        .btn:hover { background: #388bfd; }
        #preview { max-width: 100%; border-radius: 8px; margin-top: 12px; display: none; }
        #result-img { max-width: 100%; border-radius: 8px; display: none; margin-top: 10px; }
        .stats { display: flex; gap: 12px; margin-bottom: 14px; flex-wrap: wrap; }
        .stat-box { flex: 1; background: #0d1117; border: 1px solid #30363d; border-radius: 8px; padding: 14px; text-align: center; min-width: 90px; }
        .stat-box .num { font-size: 28px; font-weight: bold; color: #58a6ff; }
        .stat-box .lbl { font-size: 11px; color: #8b949e; margin-top: 4px; }
        #map { height: 280px; border-radius: 8px; }
        .alert-box { background: #1c2a1c; border: 1px solid #3fb950; border-radius: 8px; padding: 14px; margin-top: 14px; display: none; }
        .alert-box p { color: #3fb950; font-size: 13px; }
        .herd-alert { background: #2a1c1c; border-color: #f85149; }
        .herd-alert p { color: #f85149; }
        #status { font-size: 13px; color: #8b949e; margin-top: 8px; text-align: center; }
        .location-row { display: flex; gap: 8px; margin-bottom: 10px; }
        .location-row input { flex: 1; padding: 8px 10px; background: #0d1117; border: 1px solid #30363d; border-radius: 6px; color: #e6edf3; font-size: 13px; }
    </style>
</head>
<body>
<div class="header">
    <h1>Animal Herd Detection System</h1>
    <p>Upload an image to detect animal groups using HAAR Cascade &bull; OpenStreetMap alerts for herd location</p>
</div>
<div class="container">
    <div class="left">
        <div class="card">
            <h2>Upload Image</h2>
            <div class="upload-area" onclick="document.getElementById('fileInput').click()">
                <label>Click to select an image (JPG / PNG)</label>
                <input type="file" id="fileInput" accept="image/*" onchange="previewImage(event)">
            </div>
            <img id="preview" alt="Preview">
            <div class="location-row" style="margin-top:12px">
                <input type="number" id="lat" placeholder="Latitude  (e.g. 33.72)" step="any">
                <input type="number" id="lng" placeholder="Longitude (e.g. 73.04)" step="any">
            </div>
            <button class="btn" onclick="detectHerd()">Detect Herd</button>
            <p id="status">No image analysed yet.</p>
        </div>
        <div class="card">
            <h2>Detection Result</h2>
            <div class="stats">
                <div class="stat-box"><div class="num" id="count">0</div><div class="lbl">Animals</div></div>
                <div class="stat-box"><div class="num" id="herd-status">—</div><div class="lbl">Herd Status</div></div>
                <div class="stat-box"><div class="num" id="conf">—</div><div class="lbl">Threshold</div></div>
            </div>
            <img id="result-img" alt="Detection Result">
        </div>
    </div>
    <div class="right">
        <div class="card">
            <h2>Map Alert (OpenStreetMap)</h2>
            <div id="map"></div>
            <div class="alert-box" id="safe-alert">
                <p>✅ No herd detected at this location. Area is clear.</p>
            </div>
            <div class="alert-box herd-alert" id="herd-alert">
                <p>⚠️ HERD DETECTED! Animals spotted at this location. Alert triggered on map.</p>
            </div>
        </div>
        <div class="card">
            <h2>Detection Log</h2>
            <div id="log" style="font-size:13px; color:#8b949e; line-height:1.8">Waiting for detection...</div>
        </div>
    </div>
</div>

<script>
    var map = L.map('map').setView([30.3753, 69.3451], 5);
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
        attribution: 'OpenStreetMap'
    }).addTo(map);
    var marker = null;

    function previewImage(event) {
        var file = event.target.files[0];
        if (!file) return;
        var reader = new FileReader();
        reader.onload = function(e) {
            var img = document.getElementById('preview');
            img.src = e.target.result;
            img.style.display = 'block';
        };
        reader.readAsDataURL(file);
        document.getElementById('status').innerText = 'Image loaded. Click Detect Herd.';
    }

    function detectHerd() {
        var file = document.getElementById('fileInput').files[0];
        if (!file) { alert('Please select an image first!'); return; }
        document.getElementById('status').innerText = 'Analysing...';
        var lat = parseFloat(document.getElementById('lat').value) || 30.3753;
        var lng = parseFloat(document.getElementById('lng').value) || 69.3451;
        var formData = new FormData();
        formData.append('image', file);
        formData.append('lat', lat);
        formData.append('lng', lng);
        fetch('/detect', { method: 'POST', body: formData })
        .then(r => r.json())
        .then(data => {
            document.getElementById('count').innerText = data.animal_count;
            document.getElementById('herd-status').innerText = data.is_herd ? 'HERD' : 'NONE';
            document.getElementById('conf').innerText = data.threshold;
            document.getElementById('status').innerText = data.message;
            if (data.result_image) {
                var ri = document.getElementById('result-img');
                ri.src = 'data:image/jpeg;base64,' + data.result_image;
                ri.style.display = 'block';
            }
            updateMap(data.lat, data.lng, data.is_herd, data.animal_count);
            updateLog(data);
        })
        .catch(() => { document.getElementById('status').innerText = 'Error during detection.'; });
    }

    function updateMap(lat, lng, isHerd, count) {
        map.setView([lat, lng], 10);
        if (marker) map.removeLayer(marker);
        var color = isHerd ? 'red' : 'green';
        var icon = L.divIcon({
            className: '',
            html: '<div style="background:' + color + ';width:20px;height:20px;border-radius:50%;border:3px solid white;box-shadow:0 0 8px ' + color + '"></div>',
            iconSize: [20, 20]
        });
        marker = L.marker([lat, lng], {icon: icon}).addTo(map);
        var popupText = isHerd
            ? '<b>HERD DETECTED</b><br>' + count + ' animals spotted'
            : '<b>Clear</b><br>No herd at this location';
        marker.bindPopup(popupText).openPopup();
        document.getElementById('safe-alert').style.display = isHerd ? 'none' : 'block';
        document.getElementById('herd-alert').style.display = isHerd ? 'block' : 'none';
    }

    function updateLog(data) {
        var now = new Date().toLocaleTimeString();
        var log = document.getElementById('log');
        log.innerHTML = [
            '[' + now + '] Detection completed',
            'Animals detected: ' + data.animal_count,
            'Herd status: ' + (data.is_herd ? 'YES - HERD ALERT' : 'No herd'),
            'Location: (' + data.lat.toFixed(4) + ', ' + data.lng.toFixed(4) + ')',
            'Map alert: ' + (data.is_herd ? 'RED marker placed' : 'GREEN marker placed')
        ].join('<br>');
    }
</script>
</body>
</html>
'''

@app.route('/')
def index():
    return render_template_string(HTML_PAGE)

@app.route('/detect', methods=['POST'])
def detect():
    file = request.files.get('image')
    lat  = float(request.form.get('lat', 30.3753))
    lng  = float(request.form.get('lng', 69.3451))

    img_array = np.frombuffer(file.read(), np.uint8)
    img       = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    gray      = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    detections = body_cascade.detectMultiScale(
        gray,
        scaleFactor=1.05,
        minNeighbors=3,
        minSize=(30, 30)
    )

    animal_count = len(detections) if len(detections) > 0 else 0
    HERD_THRESHOLD = 3
    is_herd = animal_count >= HERD_THRESHOLD

    for (x, y, w, h) in (detections if animal_count > 0 else []):
        color = (0, 0, 255) if is_herd else (0, 255, 0)
        cv2.rectangle(img, (x, y), (x+w, y+h), color, 2)
        cv2.putText(img, 'Animal', (x, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    label = f'HERD DETECTED: {animal_count} animals' if is_herd else f'Animals: {animal_count}'
    cv2.putText(img, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255) if is_herd else (0,255,0), 2)

    _, buffer  = cv2.imencode('.jpg', img)
    result_b64 = base64.b64encode(buffer).decode('utf-8')

    return jsonify({
        'animal_count': animal_count,
        'is_herd':      is_herd,
        'threshold':    HERD_THRESHOLD,
        'lat':          lat,
        'lng':          lng,
        'message':      f'Done! {animal_count} object(s) detected. {"HERD ALERT!" if is_herd else "No herd."}',
        'result_image': result_b64
    })

def run_flask():
    app.run(debug=False, port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

print('Animal Herd Detection App is running!')
print('Open in browser: http://127.0.0.1:5000')


Animal Herd Detection App is running!
Open in browser: http://127.0.0.1:5000


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
